# Reproducing the figures

Two steps, using only the model responses already in this repo:

1. Compute the intermediate statistics (CPS, flip rate, refusal rates, ...) from the response
   data under `data/responses/`, using the scripts in `evaluate/`.
2. Plot every figure that data supports.

`evaluate/ALL_FIGURES.ipynb` has the full set of figures, including a few that need raw
experiment data not included here (noted inline there).

Run this notebook from the repository root.

In [ ]:
import os
import sys

sys.path.insert(0, ".")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import seaborn as sns
import json
from matplotlib.lines import Line2D

from src.config import PATH_ANALYSIS, PATH_RESULTS
from evaluate.plotting_config import MODEL_MAPPINGS, VARIATION_MAPPINGS
from evaluate.bootstrap_utils import bootstrap_ci

from evaluate.compute_marginal_action_likelihoods import compute_marginal_action_likelihoods
from evaluate.compute_refusal_invalid_stats import compute_refusal_invalid_stats
from evaluate.compute_cps_statistics import compute_cps_statistics, compute_cps_statistics_avg
from evaluate.compute_flip_boundary_mass import compute_flip_boundary_mass
from evaluate.compute_rule_based_cps import compute_rule_based_cps

os.makedirs(PATH_ANALYSIS, exist_ok=True)
os.makedirs("figures", exist_ok=True)

## Part 1: Generate data

Every experiment directory under `data/responses/` that holds real per-model response CSVs
(as written by `src.evaluate` + `src.collect`).

In [ ]:
RESPONSE_DIRS = [
    f"{PATH_RESULTS}/openai_models/high",
    f"{PATH_RESULTS}/anthropic_models/high",
    f"{PATH_RESULTS}/together_models/high",
    f"{PATH_RESULTS}/llama_models/high",
    f"{PATH_RESULTS}/mistral_models/high",
    f"{PATH_RESULTS}/qwen_models/high",
    f"{PATH_RESULTS}/deepseek_models/high",
]
RESPONSE_DIRS = [d for d in RESPONSE_DIRS if os.path.isdir(d)]
RESPONSE_DIRS

**Master marginal action likelihoods / CPS table** (`evaluate/compute_marginal_action_likelihoods.py`)

In [ ]:
df = pd.concat(
    [compute_marginal_action_likelihoods(d) for d in RESPONSE_DIRS], ignore_index=True
)
df.to_csv(f"{PATH_ANALYSIS}/marginal_action_likelihoods.csv", index=False)
print(f"{len(df)} rows, {df['model_id'].nunique()} models")
df.head()

**Refusal/invalid rates per model** (`evaluate/compute_refusal_invalid_stats.py`)

In [ ]:
df_refusal_invalid = pd.concat(
    [compute_refusal_invalid_stats(d) for d in RESPONSE_DIRS], ignore_index=True
)
df_refusal_invalid.to_csv(f"{PATH_ANALYSIS}/refusal_invalid_stats.csv", index=False)

# sort by MODEL_MAPPINGS order to match the plotting cells below
order = list(MODEL_MAPPINGS.keys())
sorter = dict(zip(order, range(len(order))))
df_refusal_invalid["model_id_rank"] = df_refusal_invalid["model_id"].map(sorter)
df_refusal_invalid.sort_values("model_id_rank", inplace=True)
df_refusal_invalid.drop(columns="model_id_rank", inplace=True)
df_refusal_invalid

**CPS statistics per (model, variation)**, plus the pooled-across-models average row (`evaluate/compute_cps_statistics.py`)

In [ ]:
df_statistics = compute_cps_statistics(df)
df_statistics.to_csv(f"{PATH_ANALYSIS}/cps_statistics.csv", index=False)

df_statistics_avg = compute_cps_statistics_avg(df)
df_statistics_avg.to_csv(f"{PATH_ANALYSIS}/cps_statistics_avg.csv", index=False)

df_statistics.head()

**Flip rate / boundary mass per (model, variation)** (`evaluate/compute_flip_boundary_mass.py`)

In [ ]:
df_flips = compute_flip_boundary_mass(df)
df_flips.to_csv(f"{PATH_ANALYSIS}/df_with_flips_and_boundary_mass.csv", index=False)
df_flips.head()

**Rule-based CPS scores** (`evaluate/compute_rule_based_cps.py`)

In [ ]:
scenarios_df = pd.read_csv("data/scenarios/variations_moralchoice_high_ambiguity.csv")
df_rules = compute_rule_based_cps(df, scenarios_df)
df_rules.to_csv(f"{PATH_ANALYSIS}/rule_based_cps_scores.csv", index=False)
df_rules.head()

The steering, system-prompt, benchmark, and layer-selection scripts
(`evaluate/compute_ab_vector_steering_cps.py`, `evaluate/compute_system_prompt_cps.py`,
`evaluate/compute_benchmark_results.py`, `evaluate/analyze_layers.py`) each need raw experiment
data (activation-steering responses, system-prompt-steering responses, lm-eval-harness
benchmark JSONs, extracted activations) that isn't included here, so we skip rerunning them.
The steering figures below load the tables already checked in under
`evaluate/data/ab_vector_steering_dfs/` instead.

## Part 2: Plot

### Figure 1: base preferences

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 16,
    "legend.frameon": False,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "grid.alpha": 0.25,
    "grid.color": ".6",
    "figure.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "cm",
    "mathtext.rm": "serif",
})

# ------------------------------------------------------------------
# 1. Define group structure + spacing
# ------------------------------------------------------------------
group_sizes = [4, 4, 4, 3, 3, 4]
gap = 0.5

xpos = []
current = 0.0
for g in group_sizes:
    xpos.extend(current + np.arange(g))
    current += g + gap

xpos = np.array(xpos)

# ------------------------------------------------------------------
# 2. Prepare data
# ------------------------------------------------------------------
plot_df = df.copy()
# family-grouped order: llama, mistral, qwen, deepseek, anthropic, openai (matches MODEL_MAPPINGS)
order = [m for m in MODEL_MAPPINGS.keys() if m in plot_df["model_id"].unique()]
palette = [MODEL_MAPPINGS.get(mid, {}).get("display_color", "grey") for mid in order]

# ------------------------------------------------------------------
# 3. Plot manually with proper spacing
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(15, 5))

box_width = 0.1

for i, model_id in enumerate(order):
    data = plot_df[plot_df["model_id"] == model_id]["p_action2_base"].values

    parts = ax.violinplot(
        [data],
        positions=[xpos[i]],
        widths=0.85,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )

    for pc in parts['bodies']:
        pc.set_facecolor(palette[i])
        pc.set_alpha(0.95)
        pc.set_edgecolor('black')
        pc.set_linewidth(2)

    q1, median, q3 = np.percentile(data, [25, 50, 75])

    box = plt.Rectangle(
        (xpos[i] - box_width/2, q1),
        box_width,
        q3 - q1,
        facecolor='black',
        edgecolor='black',
        linewidth=2,
        zorder=4
    )
    ax.add_patch(box)

    ax.scatter(
        xpos[i],
        median,
        s=25,
        marker="o",
        facecolors="white",
        edgecolors="black",
        linewidths=1.1,
        zorder=5
    )

# ------------------------------------------------------------------
# 4. Axes, ticks, labels
# ------------------------------------------------------------------
ax.set_xticks(xpos)
ax.set_xticklabels(
    [MODEL_MAPPINGS.get(mid, {}).get("display_name", str(mid)) for mid in order],
    rotation=45,
    ha="right"
)

ax.set_ylabel("Marginal Action Likelihood \n P(rule-violating action)", labelpad=8)
ax.set_xlabel("")
ax.set_yticks([0.0, 0.25, 0.5, 0.75, 1.0])
ax.set_ylim(-0.05, 1.05)

ax.axhline(0.5, linestyle="--", linewidth=2, color="black", alpha=0.7, zorder=1)

# ------------------------------------------------------------------
# 5. Visual separators between model families
# ------------------------------------------------------------------
boundaries = np.cumsum(group_sizes)[:-1]
sep_positions = []

for boundary_idx in boundaries:
    sep_pos = (xpos[boundary_idx - 1] + xpos[boundary_idx]) / 2
    sep_positions.append(sep_pos)
    ax.axvline(sep_pos, color='black', linestyle='-', linewidth=2, alpha=0.35)

# ------------------------------------------------------------------
# 6. Final styling
# ------------------------------------------------------------------
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(axis="both", which="major", direction="out", length=6, width=1.5)
ax.tick_params(axis="both", which="minor", direction="in", length=3, width=1)

ax.grid(axis="y", visible=False)
ax.grid(axis="x", visible=False)

ax.axhspan(-0.5, 0.25, color="#f4cccc", alpha=0.3, zorder=-3)

ax.set_xlim(xpos[0] - 0.75, xpos[-1] + 0.75)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout(pad=0.7)
plt.savefig("figures/llm_base_mal_violin.pdf", bbox_inches='tight')
plt.show()

### Figure 2: CPS scores

In [ ]:
sns.set_theme(style="ticks", font_scale=1.0)  # reduced from 1.2
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 14,        # reduced from 19
    "legend.frameon": False,
    "xtick.labelsize": 13,       # reduced from 19
    "ytick.labelsize": 12,       # reduced from 17
    "errorbar.capsize": 3,
    "grid.alpha": 0.25,
    "figure.dpi": 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif'
})

fig, axes = plt.subplots(1, 3, figsize=(11, 7.1), sharex=False, sharey=True)

variations = ["Consequentialist", "Emotional", "Relational"]

for ax, variation in zip(axes, variations):

    sub = df_statistics[df_statistics["variation"] == variation].copy()
    # family-grouped order, llama at top to openai at bottom (matches MODEL_MAPPINGS, reversed)
    model_order = [m for m in MODEL_MAPPINGS.keys() if m in sub["model_id"].values][::-1]
    sub = sub.set_index("model_id").loc[model_order]

    gap_after = [3, 6, 9, 13, 17]
    gap_size = 0.3
    ypos = []
    current_y = 0.0

    for i in range(len(sub)):
        ypos.append(current_y)
        current_y += 1.0
        if i in gap_after:
            current_y += gap_size

    ypos = np.array(ypos)
    ax.set_yticks(ypos)
    ax.set_yticklabels([MODEL_MAPPINGS[m]["display_name"] for m in sub.index], fontsize=12)

    ax.axvline(0, linestyle='--', color='gray', linewidth=2)

    for i, ((model_id, row), y) in enumerate(zip(sub.iterrows(), ypos)):
        color = MODEL_MAPPINGS[model_id]["display_color"]
        ax.errorbar(
            x=row["ci_mean"], y=y,
            xerr=[[row["ci_mean"] - row["ci_lower"]],
                  [row["ci_upper"] - row["ci_mean"]]],
            fmt='o',
            markersize=8,
            ecolor=color,
            color=color,
            elinewidth=2.5,
        )


    # add average row across models for the current variation
    avg_row = df_statistics_avg.set_index("variation").loc[variation]

    y_avg = -1.3

    ax.errorbar(
        x=avg_row["ci_mean"], y=y_avg,
        xerr=[[avg_row["ci_mean"] - avg_row["ci_lower"]],
              [avg_row["ci_upper"] - avg_row["ci_mean"]]],
        fmt='o',
        markersize=8,
        ecolor='#6B6B6B',
        color='#6B6B6B',
        elinewidth=2.5,
    )

    ax.set_yticks(list(ypos) + [y_avg])
    labels = ax.set_yticklabels(
        [MODEL_MAPPINGS[m]["display_name"] for m in sub.index] + ["Average"],
        fontsize=12,
    )
    if labels:
        labels[-1].set_fontweight('bold')
    ax.set_ylim(-2.2, max(ypos) + 0.6)
    ax.tick_params(axis="both", which="major", direction="out", length=5, width=1.5)
    ax.tick_params(axis="both", which="minor", direction="in", length=3, width=1)

    ax.text(0.085, max(ypos) + 2.0, variation, fontsize=15, fontweight='bold',
            color=VARIATION_MAPPINGS[variation]['display_color'],
            ha='center', va='center', transform=ax.transData)

    separators = [ypos[3] + 0.55, ypos[6] + 0.55, ypos[9] + 0.55, ypos[13] + 0.55, ypos[17] + 0.55, y_avg + 0.55]
    for sep in separators:
        ax.axhline(y=sep, color='black', linestyle='-', linewidth=2, alpha=0.35)

    ax.grid(axis="x", linestyle='-', linewidth=0.6, alpha=0.5)
    ax.set_xlim(-0.01, 0.17)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    color = VARIATION_MAPPINGS[variation]['display_color']
    var_letter = variation[0].upper()

    # X-axis label block
    lab_y = -4.5
    ax.text(0.06,   lab_y,        "CPS", size=15, weight="bold", ha="right", transform=ax.transData)
    ax.text(0.06,   lab_y + 0.4,  "(", size=11, ha="left", transform=ax.transData)
    ax.text(0.071,  lab_y + 0.4,  ")", size=11, ha="left", transform=ax.transData)
    ax.text(0.0635, lab_y + 0.35, var_letter, size=11, weight="bold", color=color, ha="left", transform=ax.transData)
    ax.text(0.08,   lab_y + 0.1,  "(95% CI)", size=15, ha="left", transform=ax.transData)

plt.tight_layout(pad=1.0)
plt.savefig("figures/cps_plot.pdf", dpi=300, bbox_inches='tight')
plt.show()

### Figure 3: boundary mass vs flip rate

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 24,
    "legend.frameon": False,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "errorbar.capsize": 3,
    "grid.alpha": 0.25,
    'grid.color': '.6',
    "figure.dpi": 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif'
})

plot_df = df_statistics.copy()
titles = ["Consequentialist", "Emotional", "Relational"]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True, constrained_layout=True)

def get_style(model_id):
    info = MODEL_MAPPINGS.get(model_id, None)
    if info is None:
        return dict(label=model_id, color="grey", marker="o")
    return dict(label=info["display_name"],
                color=info["display_color"],
                marker=info["marker"])

variations = list(VARIATION_MAPPINGS.keys())

for i, (ax, var) in enumerate(zip(axes, variations)):
    sub = plot_df[plot_df["variation"] == var].copy()

    for model_id in sub["model_id"].unique():
        st = get_style(model_id)
        ss = sub[sub["model_id"] == model_id]

        ax.scatter(
            ss["boundary_mass"],
            ss["flip_rate"],
            s=225,
            alpha=0.9,
            c=st["color"],
            marker=st["marker"],
            edgecolors="black",
            linewidths=1.0,
            zorder=4
        )

    ax.set_title(titles[i], fontsize=26, pad=12, color=VARIATION_MAPPINGS[titles[i]]['display_color'])

    ax.text(0.17, 0.025, "BM", size=22, weight="bold", ha="right", transform=ax.transData)
    ax.text(0.1875, 0.0215, "0.1", size=16, weight="bold", ha="right", transform=ax.transData)

    if i == 0:
        ax.set_ylabel("Flip Rate", labelpad=10)

    ax.grid(axis="y", linestyle="-", alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_xlim(0.01, 0.325)

meta_models = ['meta_llama-2-7b-chat', 'meta_llama-3-8B-instruct', 'meta_llama-3.1-8B-instruct', 'meta_llama-3.1-70b-instruct']
mistral_models = ['mistral_mixtral-8x7b-instruct_8bit', 'mistral_mistral-7b-instruct-v0.1', 'huggingfaceh4_zephyr-7b-beta', 'teknium_openhermes-2.5-mistral-7b']
qwen_models = ['qwen_qwen1.5-7b-chat', 'qwen_qwen2-7b-instruct', 'qwen_qwen3-4b-instruct', 'qwen_qwen3-8b']
deepseek_models = ['deepseek_deepseek-llm-7b-chat', 'deepseek-ai_DeepSeek-V3', 'deepseek-ai_DeepSeek-V3.1']
anthropic_models = ['claude_claude-3-haiku-20240307', 'claude_claude-haiku-4-5-20251001', 'claude_claude-sonnet-4-5-20250929']
openai_models = ['openai_gpt-4o-mini', 'openai_gpt-4.1', 'openai_gpt-4.1-mini', 'openai_gpt-5.1']

column_x_positions = [0.00, 0.185, 0.405, 0.565, 0.76, 0.915]

y_header = 1.05
y_header_offset = 0.15
row_height = 0.18

leg_ax = fig.add_axes([0.05, -0.35, 0.9, 0.3])
leg_ax.set_xlim(0, 1)
leg_ax.set_ylim(0, 1)
leg_ax.axis('off')

company_map = {
    "Meta": meta_models,
    "Mistral": mistral_models,
    "Qwen": qwen_models,
    "DeepSeek": deepseek_models,
    "Anthropic": anthropic_models,
    "OpenAI": openai_models
}

for col_idx, (company, models) in enumerate(company_map.items()):
    curr_x = column_x_positions[col_idx]

    for row_idx, mid in enumerate(models):
        curr_y = y_header - y_header_offset - (row_idx * row_height)

        if mid not in MODEL_MAPPINGS:
            continue

        m_info = MODEL_MAPPINGS[mid]

        leg_ax.scatter(curr_x + 0.01, curr_y,
                       marker=m_info['marker'],
                       color=m_info['display_color'],
                       edgecolor='black', s=200, linewidth=0.8,
                       clip_on=False, zorder=10)

        leg_ax.text(curr_x + 0.0225, curr_y, m_info['display_name'],
                    va='center', ha='left', fontsize=18)

plt.tight_layout()
plt.savefig("figures/bm_fr.pdf", bbox_inches="tight", dpi=300)
plt.show()

### Refusal / invalid plot

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 17,
    "legend.frameon": False,
    "xtick.labelsize": 16,
    "ytick.labelsize": 12,
    "errorbar.capsize": 3,
    "grid.alpha": 0.25,
    'grid.color': '.6',
    "figure.dpi": 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif'
})

models = df_refusal_invalid['model_id'].tolist()

refusals = [v["refusal_proportion"] for _, v in df_refusal_invalid.iterrows()]
invalids = [v["invalid_proportion"] for _, v in df_refusal_invalid.iterrows()]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width/2, refusals, width, label='Refusals', edgecolor="black", color='#8E3B3B')
ax.bar(x + width/2, invalids, width, label='Invalid Responses', edgecolor="black", color='#C9B458')

ax.set_ylabel('Proportion')
ax.set_xticks(x)
ax.set_xticklabels([MODEL_MAPPINGS[m]['display_name'] for m in models], rotation=45, ha='right', fontsize=12)
ax.legend(fontsize=12)

ax.grid(True, alpha=0.25, axis='y', linestyle='-', linewidth=0.8)
ax.set_axisbelow(True)

sns.despine()
plt.tight_layout()
plt.savefig("figures/llm_refusals_invalids.pdf", dpi=300)
plt.show()

### CPS distribution plot

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 20,
    "legend.frameon": False,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "grid.alpha": 0.25,
    "grid.color": ".6",
    "figure.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "cm"
})

variations = ['Consequentialist', 'Emotional', 'Relational']

model_ids = list(MODEL_MAPPINGS.keys())
n_models = len(model_ids)

data_all = df.copy()
x_min = data_all['CPS'].min() - 0.05
x_max = data_all['CPS'].max() + 0.05

fig = plt.figure(figsize=(20, n_models * 0.9), dpi=300)
outer_gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.05)

for col_idx, variation_name in enumerate(variations):
    inner_gs = gridspec.GridSpecFromSubplotSpec(
        n_models, 1,
        subplot_spec=outer_gs[col_idx],
        hspace=-0.3
    )
    axs = [fig.add_subplot(inner_gs[i]) for i in range(n_models)]

    data = data_all[data_all['variation'] == variation_name]

    for i, model_id in enumerate(model_ids):
        ax = axs[i]
        subset = data[data['model_id'] == model_id]['CPS']
        color = mcolors.to_rgb(MODEL_MAPPINGS[model_id]['display_color'])

        if len(subset) > 1:
            sns.kdeplot(
                subset, ax=ax,
                bw_adjust=0.4,
                fill=True, alpha=1.0,
                color=color,
                linewidth=0,
                clip=(x_min, 0)
            )
            sns.kdeplot(
                subset, ax=ax,
                bw_adjust=0.4,
                fill=True, alpha=1.0,
                color=color,
                linewidth=0,
                clip=(0, x_max)
            )
            sns.kdeplot(
                subset, ax=ax,
                bw_adjust=0.4,
                fill=False,
                color='white',
                linewidth=1.5,
                clip=(x_min, x_max)
            )

            mean_val = subset.mean()
            y_top = ax.get_ylim()[1] * 0.5
            ax.plot(
                [mean_val, mean_val],
                [0, y_top],
                lw=2,
                color='black',
                linestyle='-',
                alpha=0.9,
                zorder=4,
                clip_on=True
            )

        ax.axvline(0, lw=2, color='black', linestyle='--', clip_on=True, zorder=3)

        if col_idx == 0:
            label = MODEL_MAPPINGS[model_id].get('display_name', model_id.split('/')[-1])
            ax.text(
                -0.02, 0.2,
                label,
                fontweight="bold",
                color=color,
                ha="right", va="center",
                transform=ax.transAxes,
                clip_on=False,
                fontsize=14,
                fontfamily="serif"
            )
        ax.set_xticks([-0.8, -0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1.0])
        ax.patch.set_alpha(0)
        ax.set_yticks([])
        ax.set_ylabel("")
        ax.set_xlabel("")
        ax.set_xlim(x_min, x_max)
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)

        if i < n_models - 1:
            ax.set_xticks([])
            ax.spines['bottom'].set_visible(False)
        else:
            ax.spines['bottom'].set_visible(True)
            ax.spines['bottom'].set_color("#000000")
            ax.spines['bottom'].set_linewidth(1.0)
            ax.tick_params(axis='x', labelsize=16)
            ax.set_xlabel("Scenario CPS Distribution", labelpad=8, fontsize=18)

        if i == 0:
            ax.set_title(
                variation_name,
                fontsize=25,
                pad=15,
                color=VARIATION_MAPPINGS[variation_name]['display_color']
            )

plt.tight_layout()
plt.savefig("figures/cps_distributions.pdf", bbox_inches='tight', dpi=300)
plt.show()

### Base MAL vs Var MAL

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 24,
    "legend.frameon": False,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "errorbar.capsize": 3,
    "grid.alpha": 0.25,
    'grid.color': '.6',
    "figure.dpi": 300,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif'
})

plot_df = df.copy()
plot_df = plot_df.groupby(['model_id', 'variation']).apply(lambda x: x[['p_action2_base', 'p_action2_variation']].mean()).reset_index()

titles = ["Consequentialist", "Emotional", "Relational"]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True, constrained_layout=True)

def get_style(model_id):
    info = MODEL_MAPPINGS.get(model_id, None)
    if info is None:
        return dict(label=model_id, color="grey", marker="o")
    return dict(label=info["display_name"],
                color=info["display_color"],
                marker=info["marker"])

variations = list(VARIATION_MAPPINGS.keys())

for i, (ax, var) in enumerate(zip(axes, variations)):
    sub = plot_df[plot_df["variation"] == var].copy()

    for model_id in sub["model_id"].unique():
        st = get_style(model_id)
        ss = sub[sub["model_id"] == model_id]

        ax.scatter(
            ss["p_action2_base"],
            ss["p_action2_variation"],
            s=225,
            alpha=0.9,
            c=st["color"],
            marker=st["marker"],
            edgecolors="black",
            linewidths=1.0,
            zorder=4
        )

    ax.set_title(titles[i], fontsize=26, pad=12, color=VARIATION_MAPPINGS[titles[i]]['display_color'])

    ax.set_xlabel("Base version\nP(rule-violating action)", labelpad=8, fontsize=20)

    if i == 0:
        ax.set_ylabel("Contextual variation\nP(rule-violating action)", labelpad=8, fontsize=20)

    ax.grid(axis="y", linestyle="-", alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_xlim(0.225, 0.65)
    ax.set_ylim(0.225, 0.65)

    ax.plot([0.225, 0.65], [0.225, 0.65], linestyle="--", color="black", alpha=0.7, zorder=2)

meta_models = ['meta_llama-2-7b-chat', 'meta_llama-3-8B-instruct', 'meta_llama-3.1-8B-instruct', 'meta_llama-3.1-70b-instruct']
mistral_models = ['mistral_mixtral-8x7b-instruct_8bit', 'mistral_mistral-7b-instruct-v0.1', 'huggingfaceh4_zephyr-7b-beta', 'teknium_openhermes-2.5-mistral-7b']
qwen_models = ['qwen_qwen1.5-7b-chat', 'qwen_qwen2-7b-instruct', 'qwen_qwen3-4b-instruct', 'qwen_qwen3-8b']
deepseek_models = ['deepseek_deepseek-llm-7b-chat', 'deepseek-ai_DeepSeek-V3', 'deepseek-ai_DeepSeek-V3.1']
anthropic_models = ['claude_claude-3-haiku-20240307', 'claude_claude-haiku-4-5-20251001', 'claude_claude-sonnet-4-5-20250929']
openai_models = ['openai_gpt-4o-mini', 'openai_gpt-4.1', 'openai_gpt-4.1-mini', 'openai_gpt-5.1']

column_x_positions = [0.00, 0.185, 0.405, 0.565, 0.76, 0.915]

y_header = 1.0
y_header_offset = 0.15
row_height = 0.18

leg_ax = fig.add_axes([0.05, -0.35, 0.9, 0.3])
leg_ax.set_xlim(0, 1)
leg_ax.set_ylim(0, 1)
leg_ax.axis('off')

company_map = {
    "Meta": meta_models,
    "Mistral": mistral_models,
    "Qwen": qwen_models,
    "DeepSeek": deepseek_models,
    "Anthropic": anthropic_models,
    "OpenAI": openai_models
}

for col_idx, (company, models) in enumerate(company_map.items()):
    curr_x = column_x_positions[col_idx]

    for row_idx, mid in enumerate(models):
        curr_y = y_header - y_header_offset - (row_idx * row_height)

        m_info = MODEL_MAPPINGS[mid]

        leg_ax.scatter(curr_x + 0.01, curr_y,
                       marker=m_info['marker'],
                       color=m_info['display_color'],
                       edgecolor='black', s=200, linewidth=0.8,
                       clip_on=False, zorder=10)

        leg_ax.text(curr_x + 0.0225, curr_y, m_info['display_name'],
                    va='center', ha='left', fontsize=18)

plt.tight_layout()
plt.savefig("figures/mal_base_mal_var.pdf", bbox_inches="tight", dpi=300)
plt.show()

### Steering plot

Loads the activation-steering CPS bootstrap tables checked in under
`evaluate/data/ab_vector_steering_dfs/` (the raw activation-steering response data isn't
included in this repo).

In [ ]:
var_cps_bootstrap_df = pd.read_csv(PATH_ANALYSIS / "ab_vector_steering_dfs/var_bootstrap.csv")
base_cps_bootstrap_df = pd.read_csv(PATH_ANALYSIS / "ab_vector_steering_dfs/base_bootstrap.csv")

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 20,
    "legend.frameon": False,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "grid.alpha": 0.25,
    "grid.color": ".6",
    "figure.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "cm",
})

variations = ['Consequentialist', 'Emotional', 'Relational']
variation_markers = {'Consequentialist': 's', 'Emotional': 'o', 'Relational': '^'}


def plot_panel(ax, panel_df, variation_col, highlight_side, baselines=None):
    """Plot one panel with three lines (one per variation), weighted only."""
    weighted_data = panel_df[panel_df['vector_type'] == 'weighted']

    for var in variations:
        var_data = weighted_data[weighted_data[variation_col] == var].sort_values('alpha')
        if var_data.empty:
            continue

        color = VARIATION_MAPPINGS[var]['display_color']

        if highlight_side == 'left':
            focus    = var_data['alpha'] <= 0
            nonfocus = var_data['alpha'] >= 0
        elif highlight_side == 'right':
            focus    = var_data['alpha'] >= 0
            nonfocus = var_data['alpha'] <= 0
        else:
            focus = np.full(len(var_data), True)
            nonfocus = np.full(len(var_data), False)

        for mask, ci_alpha, line_alpha, mean_alpha in [
            (focus,    0.1, 0.5, 1.0),
            (nonfocus, 0.025, 0.3, 0.35),
        ]:
            seg = var_data[mask]
            if seg.empty:
                continue

            ax.fill_between(
                seg['alpha'], seg['CPS_lower'], seg['CPS_upper'],
                color=color, alpha=ci_alpha, zorder=1
            )
            ax.plot(seg['alpha'], seg['CPS_lower'], color=color, alpha=line_alpha, linewidth=1, zorder=2)
            ax.plot(seg['alpha'], seg['CPS_upper'], color=color, alpha=line_alpha, linewidth=1, zorder=2)
            ax.plot(
                seg['alpha'], seg['CPS_mean'],
                color=color,
                marker=variation_markers[var],
                linewidth=2.5, markersize=8,
                markeredgecolor='black', markeredgewidth=0.7,
                alpha=mean_alpha, zorder=3
            )

    ax.set_xlabel('Alpha', labelpad=8)
    ax.set_ylabel(
        r'$\mathbf{CPS}^{\boldsymbol{(v)}}$ (95% CI)',
        fontsize=20, fontweight='normal'
    )
    ax.grid(axis="y", linestyle="-", alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.axhline(0, color='gray', linestyle='-', linewidth=2, alpha=0.7)


fig, axs = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

plot_panel(axs[0], var_cps_bootstrap_df,  variation_col='variation',  highlight_side='left')
plot_panel(axs[1], base_cps_bootstrap_df, variation_col='var_vector', highlight_side='right')

axs[0].set_title(
    "Attenuating contextual sensitivity:\n",
    fontsize=20, fontweight="bold", fontstyle="italic", pad=15
)
axs[1].set_title(
    "Inducing contextual sensitivity:\n",
    fontsize=20, fontweight="bold", fontstyle="italic", pad=15
)

fig.tight_layout(rect=[0, 0.08, 1, 1])

handles = [
    Line2D([0], [0], color=VARIATION_MAPPINGS[var]['display_color'], lw=3, marker=variation_markers[var], markersize=7)
    for var in variations
]
fig.legend(
    handles, variations,
    loc='lower center',
    bbox_to_anchor=(0.53, -0.025),
    ncol=3,
    fontsize=20,
    frameon=False
)

plt.savefig("figures/cps_steering_main.pdf", dpi=300, bbox_inches='tight')
plt.show()

### Steering distribution plot

In [ ]:
base_cps = pd.read_csv(PATH_ANALYSIS / "ab_vector_steering_dfs/base.csv")
var_cps = pd.read_csv(PATH_ANALYSIS / "ab_vector_steering_dfs/var.csv")

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 20,
    "legend.frameon": False,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "grid.alpha": 0.25,
    "grid.color": ".6",
    "figure.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "cm"
})

variations = ['Consequentialist', 'Emotional', 'Relational']

data_all = var_cps[(var_cps['vector_type'] == 'weighted')].copy()
data_all['alpha'] = data_all['alpha'].astype(float)
alphas = sorted(data_all['alpha'].unique())
n = len(alphas)

x_min = data_all['CPS'].min() - 0.05
x_max = data_all['CPS'].max() + 0.05

fig = plt.figure(figsize=(20, 8), dpi=300)
outer_gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.15)

for col_idx, variation_name in enumerate(variations):
    inner_gs = gridspec.GridSpecFromSubplotSpec(n, 1, subplot_spec=outer_gs[col_idx], hspace=-0.15)
    axs = [fig.add_subplot(inner_gs[i]) for i in range(n)]

    data = data_all[data_all['variation'] == variation_name]
    base_color = mcolors.to_rgb(VARIATION_MAPPINGS[variation_name]['display_color'])

    def make_palette(base_color, alphas):
        color_map = {}
        for a in alphas:
            if a > 0:
                color_map[a] = (*base_color, 0.35)
            else:
                color_map[a] = (*base_color, 1.0)
        return color_map

    color_map = make_palette(base_color, alphas)

    for i, alpha_val in enumerate(alphas):
        ax = axs[i]
        subset = data[data['alpha'] == alpha_val]['CPS']
        color = color_map[alpha_val]

        fill_alpha = 0.35 if alpha_val > 0 else 1.0

        sns.kdeplot(
            subset, ax=ax,
            bw_adjust=0.5,
            fill=True, alpha=fill_alpha,
            color=color,
            linewidth=0,
            clip=(x_min, x_max)
        )
        sns.kdeplot(
            subset, ax=ax,
            bw_adjust=0.5,
            fill=False,
            color='white',
            linewidth=2,
            alpha=fill_alpha,
            clip=(x_min, x_max)
        )

        mean_val = subset.mean()
        y_top = ax.get_ylim()[1] * 0.85
        ax.plot(
            [mean_val, mean_val],
            [0, y_top],
            lw=2,
            color='black',
            linestyle=':',
            alpha=0.9,
            zorder=4,
            clip_on=True
        )

        ax.axvline(0, lw=2, color='black', linestyle='--', clip_on=True)

        if col_idx == 0:
            ax.text(
                -0.12, 0.2,
                f"\u03b1 = {alpha_val:.1f}",
                fontweight="bold",
                color="black",
                ha="right", va="center",
                transform=ax.transAxes,
                clip_on=False,
                fontsize=17,
                fontfamily="serif"
            )

        ax.patch.set_alpha(0)
        ax.set_yticks([])
        ax.set_ylabel("")
        ax.set_xlabel("")
        ax.set_xlim(x_min, x_max)
        ax.spines['left'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['top'].set_visible(False)

        if i < n - 1:
            ax.set_xticks([])
            ax.spines['bottom'].set_visible(False)
        else:
            ax.spines['bottom'].set_visible(True)
            ax.spines['bottom'].set_color("#000000")
            ax.spines['bottom'].set_linewidth(1.0)
            ax.tick_params(axis='x', labelsize=16)
            ax.set_xlabel("Scenario CPS Distribution", labelpad=8, fontsize=18)

        if i == 0:
            ax.set_title(
                variation_name,
                fontsize=20,
                pad=10,
                color=VARIATION_MAPPINGS[variation_name]['display_color']
            )

plt.show()

### Steering configuration plot

In [ ]:
cps_config_results = json.load(open(PATH_ANALYSIS / "ab_vector_steering_dfs/cps_config_data.json", "r"))

In [ ]:
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams.update({
    "axes.edgecolor": "#000000",
    "axes.linewidth": 1.0,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.labelsize": 20,
    "legend.frameon": False,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "grid.alpha": 0.25,
    "grid.color": ".6",
    "figure.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "cm"
})

variations = ['Consequentialist', 'Emotional', 'Relational']

GROUPS = [
    ("AB + Compare + Repeat",  "#E66101", "D", ["ab_comp_rep_vector_weighted",   "ab_comp_rep_vector_unweighted"]),
    ("AB",             "#4A90D9", "o", ["ab_vector_weighted",               "ab_vector_unweighted"]),
    ("Compare",        "#7B6FD6", "^", ["comp_vector_weighted",             "comp_vector_unweighted"]),
    ("Repeat",         "#C06C84", "P", ["rep_vector_weighted",              "rep_vector_unweighted"]),
    ("AB + Compare",     "#2AAA6F", "s", ["ab_comp_vector_weighted",          "ab_comp_vector_unweighted"]),
    ("AB + Repeat",      "#D4A24C", "X", ["ab_rep_vector_weighted",           "ab_rep_vector_unweighted"]),
    ("Compare + Repeat", "#888888", "v", ["comp_rep_vector_weighted",         "comp_rep_vector_unweighted"]),
]

WEIGHT_STYLES = {
    "weighted":   {"linestyle": "-"},
    "unweighted": {"linestyle": ":"},
}

def get_weight_tag(key):
    return "weighted" if key.endswith("weighted)") or key.endswith("_weighted") else "unweighted"

key_to_style = {}
for group_label, color, marker, keys in GROUPS:
    for k in keys:
        if k in cps_config_results:
            w = get_weight_tag(k)
            key_to_style[k] = {
                "color":     color,
                "linestyle": WEIGHT_STYLES[w]["linestyle"],
                "marker":    marker,
            }

row_titles = [
    r"Attenuating contextual sensitivity: ",
    r"Inducing contextual sensitivity: "
]


def make_plot(data_key, fig_title):
    fig, axs = plt.subplots(2, 3, figsize=(22, 12), sharey='row', sharex=False)

    all_keys = list(cps_config_results.keys())

    for row_idx, cps_type in enumerate(['var_cps', 'base_cps']):
        for col_idx, variation in enumerate(variations):
            ax = axs[row_idx, col_idx]

            for vec_key in all_keys:
                if vec_key not in key_to_style:
                    continue
                vec_data = cps_config_results[vec_key][data_key][cps_type]
                if variation not in vec_data:
                    continue

                style = key_to_style[vec_key]
                y_vals = np.array(vec_data[variation])
                x_vals = np.linspace(-5.0, 5.0, len(y_vals))

                if cps_type == 'var_cps':
                    focus_mask    = x_vals <= 0
                    nonfocus_mask = x_vals >= 0
                else:
                    focus_mask    = x_vals >= 0
                    nonfocus_mask = x_vals <= 0

                for mask, alpha_val in [(focus_mask, 1.0), (nonfocus_mask, 0.34)]:
                    x_seg = x_vals[mask]
                    y_seg = y_vals[mask]
                    if len(x_seg) == 0:
                        continue
                    ax.plot(
                        x_seg, y_seg,
                        color=style["color"],
                        linestyle=style["linestyle"],
                        marker=style["marker"],
                        linewidth=2.2,
                        markersize=7,
                        markeredgecolor='black',
                        markeredgewidth=0.6,
                        alpha=alpha_val,
                        zorder=3,
                    )

            if row_idx == 0:
                ax.set_title(variation, fontsize=22, pad=60, color=VARIATION_MAPPINGS[variation]['display_color'])

            if col_idx == 0:
                ax.set_ylabel(r'$\mathbf{CPS}^{\boldsymbol{(v)}}$', fontsize=20, fontweight='normal')

            ax.set_xlabel('Alpha', labelpad=8)
            ax.grid(axis="y", linestyle="-", alpha=0.2)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.axhline(0, color='gray', linestyle='-', linewidth=1.5, alpha=0.7)

    fig.text(0.42125, 0.92, row_titles[0], ha="center", va="top",
             fontsize=22, fontweight="bold", fontstyle="italic")
    fig.text(0.42125, 0.51, row_titles[1], ha="center", va="top",
             fontsize=22, fontweight="bold", fontstyle="italic")

    fig.tight_layout(rect=[0, 0.13, 1, 0.95])
    fig.subplots_adjust(top=0.875, bottom=0.18, hspace=0.40)

    n_groups = len(GROUPS)

    legend_bottom = -0.06
    legend_top    = legend_bottom + 0.055
    header_y      = legend_top   + 0.045

    x_start = 0.125
    x_end   = 0.9
    col_positions = [x_start + i * (x_end - x_start) / (n_groups - 1) for i in range(n_groups)]

    fig.text(0.53, legend_top + 0.085, "Steering Vector Source",
             ha='center', va='center',
             fontsize=24, fontweight='bold', color='black', fontstyle='italic',
             transform=fig.transFigure)

    for i, (group_label, color, marker, keys) in enumerate(GROUPS):
        x = col_positions[i]

        fig.text(x, header_y, group_label,
                 ha='center', va='center',
                 fontsize=22, fontweight='bold', color='black',
                 transform=fig.transFigure)

        for row_y, linestyle, label in [
            (legend_top,    '-',  'Weighted'),
            (legend_bottom, ':', 'Unweighted'),
        ]:
            ax_inset = fig.add_axes([x - 0.045, row_y - 0.012, 0.06, 0.024])
            ax_inset.set_axis_off()
            ax_inset.plot([0.05, 0.55], [0.5, 0.5],
                          color=color, linestyle=linestyle,
                          linewidth=2.2, marker=marker,
                          markersize=10, markeredgecolor='black',
                          markeredgewidth=0.6,
                          transform=ax_inset.transAxes,
                          clip_on=False)
            fig.text(x + 0.0, row_y, label,
                     ha='left', va='center',
                     fontsize=20, color='black',
                     transform=fig.transFigure)

    plt.show()


make_plot('last_token',    'Last Token Position')
make_plot('all_positions', 'All Positions')